In [ ]:
import os

import rootutils

root = rootutils.setup_root(os.path.abspath(""), dotenv=True, pythonpath=True, cwd=True)

from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path

import numpy as np
import polars as pl
from dateutil.relativedelta import relativedelta
from nested_ragged_tensors.ragged_numpy import JointNestedRaggedTensorDict
from omegaconf import DictConfig


@dataclass
class DummyConfig:
    """Dummy configuration for testing MEDS dataset"""

    schema_files_root: str
    task_label_path: str
    data_dir: str
    task_name: str = "dummy_task"
    max_seq_len: int = 64
    do_prepend_static_data: bool = False
    postpend_eos_token: bool = False
    do_flatten_tensors: bool = True
    EOS_TOKEN_ID: int = 5
    do_include_subject_id: bool = True
    do_include_subsequence_indices: bool = True
    do_include_start_time_min: bool = True
    do_include_end_time: bool = True
    do_include_prediction_time: bool = True
    subsequence_sampling_strategy: str = "from_start"
    code_metadata_fp: str = field(init=False)
    token_bin_size: int = 8
    token_insertion_strategy: str = "token_count"
    H_TOKEN: int = 4
    O_TOKEN: int = 5
    vocab_size: int = 6

    def __post_init__(self):
        self.code_metadata_fp = self.data_dir + "/metadata.parquet"


def create_dummy_dataset(
    base_dir: str | Path, n_subjects: int = 3, split: str = "train", seed: int | None = 42, n_repeats: int = 3
) -> DummyConfig:
    if seed is not None:
        np.random.seed(seed)

    base_dir = Path(base_dir)

    # Create directories
    schema_dir = base_dir / "schema" / split
    schema_dir.mkdir(parents=True, exist_ok=True)
    base_dir.joinpath("data").mkdir(exist_ok=True)

    # Create static data
    base_datetime = datetime(1995, 1, 1)
    static_data = []
    for subject_id in range(n_subjects):
        static_data.append(
            {
                "subject_id": subject_id,
                "start_time": base_datetime,
                "time": [base_datetime + relativedelta(days=i) for i in range(8 * n_repeats)],
                "code": [1, 2, 3],
                "numeric_value": [0.1, 0.2, 0.3],
            }
        )
    static_df = pl.DataFrame(static_data)
    static_df.write_parquet(schema_dir / "shard_0.parquet", use_pyarrow=True)

    # Create dynamic data with consistent sequence lengths
    subject_dynamic_data = []
    for subject_id in range(n_subjects):
        rand_n_repeats = np.random.randint(8, 8 * n_repeats)
        dynamic_data = JointNestedRaggedTensorDict(
            raw_tensors={
                "code": ([[1], [2], [1], [2], [1], [2], [1], [3]] * rand_n_repeats),
                "numeric_value": (
                    [
                        [np.nan],
                        [np.nan],
                        [np.nan],
                        [np.nan],
                        [np.nan],
                        [np.nan],
                        [np.nan],
                        [np.nan],
                    ]
                    * rand_n_repeats
                ),
                "time_delta_days": ([1, 1, 1, 1, 1, 1, 1, 1] * rand_n_repeats),
            }
        )
        subject_dynamic_data.append(dynamic_data)
    dynamic_data = JointNestedRaggedTensorDict.vstack(subject_dynamic_data)

    nrt_output_dir = base_dir / "data" / split
    nrt_output_dir.mkdir(parents=True, exist_ok=True)
    dynamic_data.save(nrt_output_dir / "shard_0.nrt")

    # Create task labels
    task_df = pl.DataFrame(
        {
            "subject_id": list(range(n_subjects)),
            "prediction_time": [base_datetime + relativedelta(years=3)] * n_subjects,
            "boolean_value": [i % 2 for i in range(n_subjects)],
        }
    )

    task_fp = base_dir / "task_labels.parquet"
    task_df.write_parquet(task_fp, use_pyarrow=True)

    metadata_df = pl.DataFrame(
        {
            "code": ["a", "b", "c", "[H]", "[NTP]"],
            "code/vocab_index": [1, 2, 3, 4, 5],
            "values/min": [None, None, None, None, None],
            "values/max": [None, None, None, None, None],
            "values/sum": [None, None, None, None, None],
            "values/n_occurrences": [None, None, None, None, None],
            "values/quantiles": [
                {"values/quantile/0.25": None, "values/quantile/0.5": None, "values/quantile/0.75": None},
                {"values/quantile/0.25": None, "values/quantile/0.5": None, "values/quantile/0.75": None},
                {"values/quantile/0.25": None, "values/quantile/0.5": None, "values/quantile/0.75": None},
                {"values/quantile/0.25": None, "values/quantile/0.5": None, "values/quantile/0.75": None},
                {"values/quantile/0.25": None, "values/quantile/0.5": None, "values/quantile/0.75": None},
            ],
        }
    )

    config = DummyConfig(
        schema_files_root=str(base_dir / "schema"),
        task_label_path=str(task_fp),
        data_dir=str(base_dir),
    )
    assert config.vocab_size == len(metadata_df) + 1

    metadata_df.write_parquet(config.code_metadata_fp)

    return config

In [ ]:
import os
import tempfile

from meds_torch.data.components.histogram_pytorch_dataset import HistogramPytorchDataset

os.environ["CUDA_VISIBLE_DEVICES"] = "0"


tmp_dir = tempfile.TemporaryDirectory()
data_config = create_dummy_dataset(tmp_dir.name, n_subjects=64)
dataset = HistogramPytorchDataset(data_config, split="train")

dynamic_data, subject_id, st, end = dataset.load_subject_dynamic_data(0)
print("every patient has identical data:")
print(dynamic_data.flatten().to_dense()["code"])
print("the length of the data is:", str(len(dynamic_data.flatten().to_dense()["code"])))

In [ ]:
"""This file prepares config fixtures for other tests."""

from pathlib import Path

import hydra
from hydra import compose, initialize
from omegaconf import DictConfig

from meds_torch.utils.resolvers import setup_resolvers

setup_resolvers()


def create_cfg(overrides, config_name="train.yaml") -> DictConfig:
    """Helper function to create Hydra DictConfig with given overrides and common settings."""
    with initialize(version_base="1.3", config_path="../src/meds_torch/configs"):
        cfg = compose(config_name=config_name, return_hydra_config=True, overrides=overrides)
    return cfg


output_dir = Path(tmp_dir.name) / "output"
overrides = [
    "experiment=histogram_eic_forecast_mtr",
    "model=histogram_forecasting",
    "model/backbone=histogram_transformer_decoder",
    "model/input_encoder=histogram_encoder",
    "data=histogram_pytorch_dataset",
    "data.H_TOKEN=4",
    "data.O_TOKEN=5",
    f"data.vocab_size={data_config.vocab_size}",
    "trainer=gpu",
    "data.subsequence_sampling_strategy=random",
    "data.token_insertion_strategy=token_count",
    "data.token_bin_size=8",
    f"data.code_metadata_fp={data_config.code_metadata_fp}",
    "model.optimizer.lr=0.001",
    "trainer.max_epochs=5",
    f"paths.output_dir={output_dir}",
    "model.top_k_acc=[1]",
    "model.diffusion_loss.target_channels=22",
    f"hydra.searchpath=[pkg://meds_torch.configs,{root}/ZERO_SHOT_TUTORIAL/configs/]",
]
cfg = create_cfg(overrides)
model = hydra.utils.instantiate(cfg.model)
type(model)

In [ ]:
from torch.utils.data.dataloader import DataLoader

train_dataloader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=dataset.collate)
val_dataloader = DataLoader(dataset, batch_size=8, shuffle=False, collate_fn=dataset.collate)

In [ ]:
# trainer = hydra.utils.instantiate(cfg.trainer)
# trainer.fit(
#     model=model,
#     train_dataloaders=train_dataloader,
#     val_dataloaders=val_dataloader,
#     ckpt_path=cfg.get("ckpt_path"),
# )
# trainer.validate(model=model, dataloaders=val_dataloader, ckpt_path=cfg.get("ckpt_path"))

# from meds_torch.latest_dir import get_latest_directory

# print("Enter the following in terminal to view the tensorboard logs:")
# print("tensorboard --logdir=%s" % get_latest_directory(cfg.paths.output_dir) + "/lightning_logs/")

In [ ]:
# Try Generation
import torch

model.cfg.generate_id = 0
model.cfg.max_tokens_budget = 24

from meds_torch.input_encoder import INPUT_ENCODER_MASK_KEY, INPUT_ENCODER_TOKENS_KEY

batch = dataset.collate([dataset[i] for i in range(8)])
input_batch = model.input_encoder.forward(batch)
prompts, mask = input_batch[INPUT_ENCODER_TOKENS_KEY], input_batch[INPUT_ENCODER_MASK_KEY]

prompt_lengths = mask.sum(dim=-1)
prompt_lengths
output_batch = model.forward(input_batch)

in_seq = output_batch["code"]
out_seq = output_batch["GENERATE//0"].filter(pl.col("subject_id") == 0)["code/vocab_index"].to_list()
print(f"raw_out_seq: {out_seq}")
print(f"in_seq: {list(filter(lambda x: x != 0, in_seq[0].tolist()))}")
print(f"out_seq: {list(filter(lambda x: x not in [0, 4, 5], out_seq))}")

In [ ]:
# TODO: log the histogram generated by the model at each input element


# Try generation analysis:

model.cfg.generate_id = 0
mask = output_batch["mask"].to(torch.bool)[:1]
mask[:] = False
mask[0, 0] = True
code = output_batch["code"][:1][mask].unsqueeze(0)
# print(output_batch["code"][0])
histogram = output_batch["histogram"][:1][mask, :].unsqueeze(0)
mask = mask[mask].unsqueeze(0)
print(f"in_seq: {code.tolist()}")
print(f"in_code: {code.shape}")
print(f"in_histogram: {histogram.shape}")
assert mask.all().item(), "mask is not all true"
batch = dict(
    code=code,
    histogram=histogram,
    mask=mask,
    prediction_time=[datetime(2025, 1, 1)],
    end_time=[datetime(2025, 1, 1)],
    subject_id=[0],
)
(
    next_token_logits,
    next_token_histogram,
    next_token_histogram_logits,
    next_token_histogram_counts,
    last_embeddings,
) = model.get_sample(batch)

print(f"next_token_logits: {next_token_logits.tolist()}")
print(f"next_token_histogram: {next_token_histogram.tolist()}")
print(f"next_token_histogram_logits: {next_token_histogram_logits.tolist()}")
print(f"next_token_histogram_logits: {next_token_histogram_counts.tolist()}")
print(f"last_embeddings: {last_embeddings.tolist()}")


model.diffusion.sample(last_embeddings, temperature=1.0)

In [ ]:
def topk(x: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
    """
    Creates a mask where values are 0 for the top-k elements in each batch and 1 elsewhere.

    Args:
        x : tensor of shape [B, L] containing values to find top-k elements
        k : tensor of shape [B, 1] containing the number of top elements to find for each batch

    Returns:
        final_mask: tensor of shape [B,L] where final_mask[b,i] = 0 if x[b,i] is in
                   the k[b] biggest values of x[b,:], else final_mask[b,i] = 1
    """
    B, L = x.shape  # batchsize, list size

    # Get indices sorted in descending order
    _, indices_des = torch.sort(x, dim=-1, descending=True)

    # Create range mask [1, L] and repeat it B times
    mask = torch.arange(L, device=x.device).unsqueeze(0).expand(B, -1)
    k_expanded = k.expand(-1, L)
    mask = mask < k_expanded

    # Create one-hot encoding and apply mask
    one_hot = torch.nn.functional.one_hot(indices_des, num_classes=L).float()
    one_hot = one_hot * mask.unsqueeze(-1)

    # Sum along the appropriate dimension to get final mask
    final_mask = one_hot.sum(dim=1)

    # Flip the mask (0 for top-k, 1 for others)
    return final_mask


def convert_to_counts_batched(probs: torch.Tensor, N: torch.Tensor) -> torch.Tensor:
    """Convert logits to counts by flooring the probabilities and then distributing
    the remaining counts to the top decimal parts.

    Args:
        logits_batch (torch.Tensor): batch of logits of shape [B, L]
        N (torch.Tensor): number of counts to use to convert logits to a histogram,
        accepts shape [B] or [B, 1]

    Returns:
        torch.Tensor: counts of shape [B, L]

    Examples:
        >>> import torch
        >>> N = torch.tensor([5,5,5,4])
        >>> test_case = torch.tensor([
        ...     [1., 1., 1., 1., 1., 1., 1., 1.],  # All equal
        ...     [2., 1., 1., 1., 1., 1., 1., 1.],  # First value different
        ...     [2., 2., 1., 0., 0., 0., 0., 0.],  # Two twos, one one
        ...     [100., 100., 100., -100., -100., -100., -100., -100.]  # Extreme values
        ... ])
        >>> result = convert_to_counts_batched(test_case, N)
        >>> expected = torch.tensor([
        ...     [1, 1, 1, 1, 1, 0, 0, 0],
        ...     [1, 1, 1, 1, 1, 0, 0, 0],
        ...     [2, 2, 1, 0, 0, 0, 0, 0],
        ...     [2, 1, 1, 0, 0, 0, 0, 0]
        ... ])
        >>> (result == expected).all().item()
        True
        >>> (result.sum(dim=-1) == N).all().item()
        True
    """
    N = N.reshape(-1, 1)
    # Expect logits_batch shape: (batch_size, vocab_size)
    counts = torch.floor(probs * N).long()
    remaining = N - counts.sum(dim=-1, keepdim=True)  # Shape: (batch_size, 1)

    decimal_parts = (probs * N) - counts.float()
    # Get indices of top decimal parts for each batch
    top_k_decimal_mask = topk(decimal_parts, remaining).to(torch.bool)

    # Increment top k in parallel
    counts[top_k_decimal_mask] += 1

    return counts


class HistogramNormalizer(torch.nn.Module):
    def __init__(self, h_token, o_token, num_bits=16):
        super().__init__()
        self.h_token = h_token
        self.o_token = o_token
        self.num_bits = num_bits
        # Create powers of 2 as a buffer to avoid recomputing
        self.register_buffer("powers", torch.pow(2, torch.arange(num_bits - 1, -1, -1).float()))

    def count_to_binary(self, count):
        """Convert number(s) to binary representation using PyTorch operations."""
        return ((count.unsqueeze(-1) // self.powers) % 2).to(torch.int)

    def binary_to_count(self, binary):
        return (binary * self.powers).sum(dim=-1, keepdim=True)

    def transform(self, x):
        # Zero out special tokens
        x[:, self.h_token] = 0
        x[:, self.o_token] = 0

        # Get counts and normalize histogram
        counts = x.sum(dim=-1, keepdim=True)
        normalized_hist = x / counts

        # Convert counts to binary representation
        binary_counts = self.count_to_binary(counts.squeeze(-1))

        # Concatenate normalized histogram with binary count
        return torch.cat([normalized_hist, binary_counts], dim=-1)

    def reverse_transform(self, x):
        x = x.clip(min=0, max=1)
        # Split into histogram and binary count
        hist = x[:, : -(self.num_bits)]  # All but last num_bits dimensions
        binary = x[:, -(self.num_bits) :].round()  # Last num_bits dimensions

        x[:, self.h_token] = 0
        x[:, self.o_token] = 0

        # Convert binary back to count
        counts = self.binary_to_count(binary)

        # Scale histogram back up
        x = convert_to_counts_batched(hist / hist.sum(dim=-1, keepdim=True), counts)

        # Restore special tokens
        x[:, self.h_token] = 1
        x[:, self.o_token] = 1

        return x, counts


x = torch.ones(1, 6)
t = HistogramNormalizer(h_token=4, o_token=5)
output = t.transform(x)
histogram, count = t.reverse_transform(output)
assert count.item() == 4
assert (histogram == 1).all()


count = [0.0] * 16
count[-3] = 1.0
count[-2] = 1.0
t = HistogramNormalizer(h_token=4, o_token=5)
x = torch.tensor([[1, 0, 0, 1, 0, 0] + count])

histogram, count = t.reverse_transform(x)
assert count.item() == 6
print(histogram)
assert (histogram == torch.tensor([[3, 0, 0, 3, 1, 1]])).all()

In [ ]:
import torch

from meds_torch.models.histogram_forecasting import HistogramNormalizer

x = torch.tensor(
    [
        [
            -10425.8027,
            -21807.6543,
            -30422.4883,
            -3386.7688,
            14660.0918,
            495.6634,
            -25452.4844,
            20718.9727,
            14073.3291,
            23486.1211,
            23656.7617,
            40456.0820,
            -6850.5850,
            25710.4609,
            -33964.0820,
            15726.7949,
            -7856.3916,
            7575.6992,
            1627.0320,
            -19951.3809,
            -11168.9453,
            -24495.2441,
        ]
    ]
)
t = HistogramNormalizer(h_token=4, o_token=5)
print(x.shape)
t.reverse_transform(x)

In [ ]:
# print(
#     "Let's generate the future conditioned on the past 24 tokens (using a sliding window with a max sequence length of 24):"
# )
# future = model.model.generate(
#     prompts=prompts[:1, :],
#     mask=mask[:1, :],
#     get_next_token_time=None,
#     time_offset_years=None,
#     temperature=model.cfg.temperature,
#     eos_tokens=model.cfg.eos_tokens,
# )

In [ ]:
# input_batch.keys()

In [ ]:
# test_batch = dict(
#     time_delta_days=input_batch["time_delta_days"], code=input_batch["code"], mask=input_batch["mask"]
# )
# torch.functional.F.softmax(model.forward(test_batch)["MODEL//LOGITS_SEQUENCE"], dim=-1).argmax(dim=-1)

In [ ]:
# print("We observe that the future follos the repeating pattern 1,2,1,2,1,2,1,3, great!")
# print(future[0][0, :24])